# Designated area species information merging script

**This script is best used to attach species data from a CSV with both species and sitecode columns to a SHP / GPKG file with a sitecode column.**

Press shift enter to run each cell. Please navigate to the cell below using the down arrow key. Promps should appear below each cell.

In [ ]:

import csv
import geopandas as gpd
import pandas as pd
import os
import xlrd
import openpyxl
import datetime
import tkinter as tk
from tkinter.filedialog import askopenfilename
from tkinter.filedialog import askdirectory
from tkinter.filedialog import asksaveasfilename

# Function to check what encoding to use for SHP and CSV
def try_encodings(read_func, encodings=("utf-8", "latin-1", "cp1252")):
    last_error = None
    for enc in encodings:
        try:
            result = read_func(enc)
            print(f"Successfully read using encoding: {enc}")
            return result, enc
        except (UnicodeDecodeError, UnicodeError) as e:
            last_error = e
            continue
    raise ValueError(f"Could not read file with any of {encodings}. Last error: {last_error}")

# Function to allow user to select file
def request_file():
    root = tk.Tk()
    root.withdraw()
    root.attributes('-topmost', True)
    root.update()
    fn = askopenfilename(parent=root)
    root.destroy()
    return fn
    
def request_output_name():
    root = tk.Tk()
    root.attributes('-topmost', True)
    root.update()
    fn = asksaveasfilename(
        title="Save output as",
        defaultextension=".gpkg",
        filetypes=[("GeoPackage", "*.gpkg"), ("All files", "*.*")]
    )
    root.destroy()
    return fn

def get_size(file):
    size_B = os.stat(file).st_size

    if size_B < 1024**2:
        print(f"File size: {os.stat(file).st_size / 1024:.2f} kB")

    else:
        print(f"File size: {os.stat(file).st_size / 1024**2:.2f} MB")

print(f"Imports done.\nTimestamp {datetime.datetime.now()}")

# Please select the designated areas geospatial data file

**This needs to be a shapefile / geopackage file**


In [ ]:

SHP_FP = request_file()
if SHP_FP != "":
    print(f"Geospatial data file path selected:\n{SHP_FP}")
else:
    print("No file selected. Won't be able to proceed until this cell is rerun and a file is selected.")


# Please select the species information datasheet

**Can be Excel or CSV file**

In [ ]:

CSV_FP = request_file()
if CSV_FP != "":
    print(f"Species information datasheet file path selected:\n{CSV_FP}")
else:
    print("No file selected. Won't be able to proceed until this cell is rerun and a file is selected.")

# Please select the output file and name
**This is the location that you want the combined species/sitecode file to be in**

In [ ]:

OUTPUT_NAME = request_output_name()
if OUTPUT_NAME != "":
    print(f"Output location and name selected:\n{OUTPUT_NAME}")
else:
    print("No location/name selected. Won't be able to proceed until this cell is rerun and a location is selected.")


In [ ]:

################################## Reading Files ##############################
print(f"Reading {SHP_FP}\nplease wait...")
get_size(SHP_FP)

# Reading Sitecode file SHP (encoding auto-detected)
SHP, encoding_SHP = try_encodings(
    lambda enc: gpd.read_file(SHP_FP, encoding=enc)
)

ext = os.path.splitext(CSV_FP)[1].lower()
if ext == ".csv":
    print(f"Reading {CSV_FP}\nplease wait...")
    get_size(CSV_FP)
    species, encoding_CSV = try_encodings(
        lambda enc: pd.read_csv(CSV_FP, encoding=enc))
elif ext in (".xlsx", ".xlsm", ".xlsb", ".xltx", ".xltm", ".xlam", ".xls", ".xlt", ".xla", ".xlm", ".xlw"):
    print(f"Reading {CSV_FP}\nplease wait...")
    get_size(CSV_FP)
    species = pd.read_excel(CSV_FP)
    print("Successfully read using Excel encoding")
else:
    raise ValueError(f"Unsupported file type: {ext}. Please use Excel or CSV files.")

# Please select the row containing the data headers # 

In [ ]:

print(species.iloc[:8,:3])
CN_IncSkip = int(input("\nWhat row are the data headers on, based off of the row numbers on the left?\nIf they do not have a row number, input '-1'.\n"))
CN_Skip = CN_IncSkip + 1
ext = os.path.splitext(CSV_FP)[1].lower()
if ext == ".csv":
    species, encoding_CSV = try_encodings(
        lambda enc: pd.read_csv(CSV_FP, encoding=enc, skiprows=CN_Skip))
elif ext in (".xlsx", ".xlsm", ".xlsb", ".xltx", ".xltm", ".xlam", ".xls", ".xlt", ".xla", ".xlm", ".xlw"):
    species = pd.read_excel(CSV_FP, skiprows=(CN_Skip))
    print("Excel file succesfully read")
else:
    raise ValueError(f"Unsupported file type: {ext}. Please use Excel or CSV files.")
print(f"\nHeaders located on row {CN_IncSkip}. Preview:")
print(species.iloc[:8,:3])

**Please write the name of the Sitecode column in the geospatial data file (case sensitive)**

In [ ]:

print("\nSHP / GPKG columns:")
print(list(SHP.columns))


SITECODE_SHP = input("What is the name of the Sitecode column in the SHP file?\n") 


**Please write the name of the Sitecode and Species column in the species information datasheet file**

In [ ]:

#Preview of columns for CSV
print("\nCSV columns:")
print(list(species.columns))

 
SITECODE_CSV = input("\nWhat is the name of the Sitecode column in the datasheet file (CASE SENSITIVE)?\n")

SPECIES_Column = input("\nWhat is the name of the Species column in the datasheet file (CASE SENSITIVE)?\n")


**Run the next cell to create the merged file**

In [ ]:

print("Creating your new file, please wait...")
########### Grouping Species under Sitecodes ########

species_grouped = (
    species.groupby(SITECODE_CSV)[SPECIES_Column]# Matches species to their shared site4code
    
    # Converts each value to a string, removes duplicates within each group,
    # puts them in alphabetical order and combines them into comma seperated string.
    .apply(lambda names: ", ".join(sorted(set(names.astype(str))))) 
    .reset_index() # turns sitecode index back to regular dataframe with two columns
    .rename(columns={SPECIES_Column: "Species"}) # Keeps consistency by renaming species name,
                                          # ie rather than 'BIRDNAME'
)

print("Species grouped under sitecodes.")
############ Matching grouped species to SHP Sitecodes #########

SHP[SITECODE_SHP] = SHP[SITECODE_SHP].astype(str).str.strip() # Forces sitecode column to string and removes blank spaces
species_grouped[SITECODE_SHP] = species_grouped[SITECODE_CSV].astype(str).str.strip() # Creates new column in species_grouped with SITECODE_SHP header
SHP = SHP.merge(species_grouped, on=SITECODE_SHP, how='left') # Attaches corresponding species column where species_grouped sitecode and SHP sitecode match

print("Matched grouped species to geospatial file sitecodes.")
print("Species data appended to", SHP['Species'].notna().sum(), f"Out of {len(SHP)} sites.")
print("Saving file...")

############### Saving ###############

SHP.to_file(OUTPUT_NAME, driver="GPKG")

print(f"File creation successful! Your new geospatial data file has been saved to: \n{OUTPUT_NAME}\n")